In [ ]:
# parameters
# BINDER_FAST: set n_periods=5, n_t=50 for fast cloud execution
N = 8                 # Hilbert space truncation
omega0_GHz = 5.0      # bare 0->1 frequency (GHz)
K_GHz = 0.2           # Kerr nonlinearity (GHz)
epsilon_GHz = 0.02    # drive amplitude (GHz); 0.1 K — weak drive
n_periods = 20        # number of drive periods to simulate
n_t = 200             # time points per simulation


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.driven_kerr import (
    DrivenKerrConfig,
    make_H0,
    make_H_drive_td,
    make_operators,
)

TWO_PI = 2 * np.pi


## Module 3a: The Rotating Wave Approximation and the Driven Kerr Oscillator

**Learning objectives:**
- Derive the rotating-frame transformation from the lab-frame Hamiltonian step by step
- Understand when the counter-rotating (anti-RWA) terms are negligible
- Numerically compare exact lab-frame dynamics with the RWA approximation
- Observe and explain the AC Stark shift of dressed eigenfrequencies vs. drive amplitude

---

**Sections:**
[1 The Driven Kerr Oscillator](#sec1) · 
[2 Moving to the Rotating Frame](#sec2) · 
[3 Exact vs. RWA Dynamics](#sec3) · 
[4 AC Stark Shift](#sec4) · 
[5 Exercises](#sec5)


<a id="sec1"></a>
## 1  The Driven Kerr Oscillator

A microwave resonator with a Josephson junction (or SNAIL) acquires an effective
photon–photon interaction (Kerr nonlinearity) that makes the level spacing
depend on occupation number. The lab-frame Hamiltonian is

$$\hat{H}(t) = \underbrace{\omega_0\hat{a}^\dagger\hat{a}
  - \frac{K}{2}\hat{a}^\dagger\hat{a}^\dagger\hat{a}\hat{a}}_{\hat{H}_0}
  + \epsilon\cos(\omega_d t)(\hat{a} + \hat{a}^\dagger).$$

Here $\omega_0$ is the bare $|0\rangle\to|1\rangle$ frequency, $K > 0$ is the Kerr
coefficient, $\epsilon$ is the drive amplitude, and $\omega_d$ is the drive frequency.
The static part $\hat{H}_0$ has eigenvalues

$$E_n = \omega_0 n - \frac{K}{2}n(n-1), \qquad n = 0, 1, 2, \ldots$$

so the $n\to n+1$ transition frequency is

$$\omega_{n,n+1} = \omega_0 - Kn.$$

The **anharmonicity** $-K$ is what distinguishes the oscillator from a pure harmonic
resonator: adjacent transitions are separated by $K$.
A microwave drive at $\omega_d \approx \omega_0$ can selectively address the $|0\rangle\to|1\rangle$
transition without exciting higher Fock levels — the foundation of
selective-number-state (SNAP) gates in Module 4.

*Lab note: $\omega_0 / 2\pi$ in cQED is typically 4–8 GHz, $K / 2\pi$ is 100–500 MHz for
transmons and 10–200 MHz for SNAIL-based Kerr resonators.
The condition $K \ll \omega_0$ means the anharmonicity is a small fraction of the
carrier frequency — ideal for the RWA derived below.*


In [ ]:
# Build configuration and display H0 eigenvalues
cfg = DrivenKerrConfig(
    N        = N,
    omega0   = TWO_PI * omega0_GHz,
    K        = TWO_PI * K_GHz,
    omega_d  = TWO_PI * (omega0_GHz - 0.005),  # 5 MHz red-detuned
    epsilon  = TWO_PI * epsilon_GHz,
    kappa    = TWO_PI * 1e-3,
    Gamma    = TWO_PI * 0.1,
    nbar     = 0.02,
    gamma_phi= TWO_PI * 0.05e-3,
    k_max    = 5,
    n_t      = 512,
)

H0 = make_H0(cfg)
evals = H0.eigenenergies()

print("Fock level energies (units: rad·GHz = ω₀/2π = 1 GHz unit)")
print(f"{'n':>4}  {'E_n / 2π (GHz)':>18}  {'spacing ω_n→n+1 / 2π (GHz)':>30}")
for n in range(min(6, N)):
    spacing_str = ""
    if n < min(5, N-1):
        spacing = (evals[n+1] - evals[n]) / TWO_PI
        spacing_str = f"{spacing:.4f}"
    print(f"{n:>4}  {evals[n]/TWO_PI:>18.4f}  {spacing_str:>30}")

print(f"\nAnharmonicity K/2π = {K_GHz*1e3:.0f} MHz")
print("Each successive transition is red-detuned by K from the one below.")


In [ ]:
# Energy level diagram
fig, ax = plt.subplots(figsize=(3.5, 5))
for n in range(min(6, N)):
    ax.axhline(evals[n] / TWO_PI, xmin=0.15, xmax=0.75, lw=2.5, color="steelblue")
    ax.text(0.78, evals[n] / TWO_PI,
            rf"$|{n}\rangle$", va="center", fontsize=10,
            transform=ax.get_yaxis_transform())

# Annotate spacing difference (anharmonicity)
if N >= 3:
    gap01 = (evals[1] - evals[0]) / TWO_PI
    gap12 = (evals[2] - evals[1]) / TWO_PI
    ax.annotate("", xy=(0.12, evals[1]/TWO_PI), xytext=(0.12, evals[0]/TWO_PI),
                xycoords=("axes fraction", "data"),
                textcoords=("axes fraction", "data"),
                arrowprops=dict(arrowstyle="<->", color="C1", lw=1.5))
    ax.text(0.03, (evals[0]+evals[1])/(2*TWO_PI),
            r"$\omega_0$", va="center", fontsize=9,
            transform=ax.get_yaxis_transform(), color="C1")
    ax.annotate("", xy=(0.12, evals[2]/TWO_PI), xytext=(0.12, evals[1]/TWO_PI),
                xycoords=("axes fraction", "data"),
                textcoords=("axes fraction", "data"),
                arrowprops=dict(arrowstyle="<->", color="C2", lw=1.5))
    ax.text(0.03, (evals[1]+evals[2])/(2*TWO_PI),
            r"$\omega_0\!-\!K$", va="center", fontsize=8,
            transform=ax.get_yaxis_transform(), color="C2")

ax.set_ylabel(r"Energy $E_n / \hbar$ (GHz)")
ax.set_xticks([])
ax.set_title(rf"Kerr oscillator: $\omega_0/2\pi={omega0_GHz}$ GHz, $K/2\pi={K_GHz*1e3:.0f}$ MHz")
ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()


<a id="sec2"></a>
## 2  Moving to the Rotating Frame

### 2.1  Interaction picture transformation

We move to a frame rotating at the drive frequency $\omega_d$ via the unitary

$$\hat{U}_0(t) = e^{-i\omega_d\hat{n}t}, \quad \hat{n} = \hat{a}^\dagger\hat{a}.$$

Under this rotation, the ladder operators transform as

$$\hat{U}_0^\dagger \hat{a}\hat{U}_0 = \hat{a}\,e^{-i\omega_d t},
\qquad \hat{U}_0^\dagger \hat{a}^\dagger\hat{U}_0 = \hat{a}^\dagger e^{+i\omega_d t}.$$

The rotating-frame Hamiltonian is $\hat{H}' = \hat{U}_0^\dagger\hat{H}\hat{U}_0
  - i\hat{U}_0^\dagger\dot{\hat{U}}_0$.

### 2.2  Sorting the terms

Expanding the drive term $\epsilon\cos(\omega_d t)(\hat{a}+\hat{a}^\dagger)$ using
$\cos(\omega_d t) = (e^{i\omega_d t}+e^{-i\omega_d t})/2$:

$$\hat{H}'_{\rm drive}(t)
= \frac{\epsilon}{2}\bigl(\hat{a}+\hat{a}^\dagger\bigr)
+ \frac{\epsilon}{2}\bigl(\hat{a}\,e^{-2i\omega_d t}+\hat{a}^\dagger e^{+2i\omega_d t}\bigr).$$

The first line oscillates slowly (or not at all if $\Delta \equiv \omega_d - \omega_0 = 0$);
the second oscillates at $2\omega_d \gg K, \kappa$.

### 2.3  Rotating Wave Approximation (RWA)

Dropping the rapidly oscillating $e^{\pm 2i\omega_d t}$ terms — valid when
$\epsilon \ll 2\omega_d$ — leaves the **effective RWA Hamiltonian in the rotating frame**:

$$\boxed{\hat{H}_{\rm eff} = -\Delta\hat{a}^\dagger\hat{a}
  - \frac{K}{2}\hat{n}(\hat{n}-1)
  + \frac{\epsilon}{2}(\hat{a}+\hat{a}^\dagger)}$$

where $\Delta = \omega_d - \omega_0$ is the drive detuning.
This Hamiltonian is time-independent — the $\epsilon/2$ term acts like a static
displacement of the oscillator combined with the Kerr nonlinearity.

*Lab note: the RWA is excellent for microwave drives because
$\omega_d / 2\pi \approx 5\,\text{GHz}$ while $\epsilon / 2\pi \lesssim 0.5\,\text{GHz}$,
giving $\epsilon / (2\omega_d) \lesssim 5\%$.
Counter-rotating corrections appear at order $(\epsilon/2\omega_d)^2 \sim 0.25\%$.
The Floquet approach in Notebook 3c goes beyond the RWA by including
these corrections non-perturbatively.*


In [ ]:
# Inspect the time-dependent Hamiltonian structure returned by make_H_drive_td
H_td = make_H_drive_td(cfg)

print("H_td structure (QuTiP mesolve list format):")
print(f"  H_td[0] = H0  (static part), shape = {H_td[0].shape}")
print(f"  H_td[1] = [H_amp, coeff_func]")
print(f"    H_amp shape = {H_td[1][0].shape}  (a + a†, X-quadrature coupling)")
print(f"    coeff_func  = ε cos(ω_d t),  ε/2π = {cfg.epsilon/TWO_PI:.4f} GHz")
print()
print("First three rows/cols of H_amp = a + a†:")
print(np.round(H_td[1][0].full()[:4, :4].real, 4))


<a id="sec3"></a>
## 3  Exact vs. RWA Dynamics

We now compare the exact lab-frame dynamics (retaining all terms) with the
rotating-frame RWA dynamics.  Both simulations start in the vacuum $|0\rangle$;
a resonant or near-resonant drive populates the oscillator.

At weak drive ($\epsilon \ll \omega_d$):
- **RWA result:** smooth coherent population transfer, $\langle\hat{n}\rangle$ grows
  monotonically on timescale $1/\epsilon$.
- **Exact result:** same growth, plus a small rapid oscillation at $2\omega_d$ from
  the counter-rotating terms. The amplitude of the oscillation is
  $\sim (\epsilon / 2\omega_d)^2 \langle n\rangle$.

The two agree to within this small amplitude, confirming RWA validity.


In [ ]:
# --- Build operators and RWA Hamiltonian ---
a, adag, n_op = make_operators(cfg.N)

# Exact lab-frame: full time-dependent H
H_exact = make_H_drive_td(cfg)

# RWA: time-independent effective Hamiltonian in rotating frame
Delta = cfg.omega_d - cfg.omega0         # detuning (rad/s)
H_rwa = (-Delta * n_op
         - (cfg.K / 2) * adag * adag * a * a
         + (cfg.epsilon / 2) * (a + adag))

# Time grid: n_periods drive periods
T_drive = cfg.T_d                        # drive period (seconds)
t_total = n_periods * T_drive
tlist = np.linspace(0, t_total, n_t)

print(f"Drive period T_d = {T_drive * 1e9:.4f} ns")
print(f"Total simulation time = {t_total * 1e9:.2f} ns  ({n_periods} periods)")
print(f"ε/2ω_d = {cfg.epsilon / (2 * cfg.omega_d):.4f}  (RWA parameter, should be ≪ 1)")


In [ ]:
# --- Run both simulations (no dissipation — pure state dynamics) ---
rho0 = qt.fock_dm(cfg.N, 0)  # start in vacuum

print("Running exact lab-frame simulation...")
result_exact = qt.mesolve(
    H_exact, rho0, tlist, [], [n_op],
    options={"nsteps": 20000, "rtol": 1e-8},
)
n_exact = result_exact.expect[0]

print("Running RWA simulation...")
result_rwa = qt.mesolve(
    H_rwa, rho0, tlist, [], [n_op],
)
n_rwa = result_rwa.expect[0]

print("Done.")


In [ ]:
# --- Plot comparison ---
tlist_ns = tlist * 1e9  # convert to nanoseconds for readability

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

ax = axes[0]
ax.plot(tlist_ns, n_exact, lw=1.2, color="steelblue", label="Exact (lab frame)", zorder=3)
ax.plot(tlist_ns, n_rwa,   lw=1.5, color="C1", ls="--", label="RWA", zorder=2)
ax.set_ylabel(r"$\langle\hat{n}\rangle$")
ax.set_title(
    rf"Exact vs. RWA: $\varepsilon/2\pi = {epsilon_GHz*1e3:.0f}$ MHz, "
    rf"$\Delta/2\pi = {abs(Delta)/TWO_PI*1e3:.1f}$ MHz"
)
ax.legend()
ax.grid(True, alpha=0.3)

# Residual: counter-rotating oscillation
ax = axes[1]
residual = n_exact - n_rwa
ax.plot(tlist_ns, residual, lw=0.8, color="C2", label="Exact − RWA")
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("Time (ns)")
ax.set_ylabel(r"$\delta\langle\hat{n}\rangle$")
ax.set_title(
    rf"Counter-rotating residual; expected amplitude $\sim (\varepsilon/2\omega_d)^2 \approx "
    rf"{(cfg.epsilon/(2*cfg.omega_d))**2:.2e}$"
)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max |Exact - RWA| = {np.max(np.abs(residual)):.2e}")
print(f"Predicted RWA error ≈ (ε/2ω_d)² = {(cfg.epsilon/(2*cfg.omega_d))**2:.2e}")


The residual panel above shows the fast oscillation from the counter-rotating terms.
The frequency of this oscillation is $2\omega_d \approx 2 \times 5\,\text{GHz} = 10\,\text{GHz}$,
which is much faster than the drive period plotted here.
The amplitude confirms the $O((\epsilon/2\omega_d)^2)$ prediction.

> In a cQED experiment, the lab-frame fast oscillation is never observed directly —
> the measurement is performed in the rotating frame (homodyne at $\omega_d$),
> automatically implementing the RWA in hardware.


<a id="sec4"></a>
## 4  AC Stark Shift (Light Shift)

When a drive is applied, the energy levels of the oscillator are shifted
even without photon exchange — this is the **AC Stark** (or *light shift*) effect.

In the RWA Hamiltonian, the dressed eigenvalues depend on $\epsilon$.
For a harmonic oscillator (no Kerr), second-order perturbation theory gives

$$\delta\omega_0^{\rm Stark} \approx \frac{\epsilon^2/4}{\Delta}$$

for the $|0\rangle\to|1\rangle$ transition, i.e., a quadratic shift in $\epsilon/\Delta$.
The sign is positive for red-detuned drives ($\Delta < 0$), pushing the transition
frequency up.  In the presence of Kerr, higher-order terms appear.

*Lab note: the AC Stark shift is one of the main sources of qubit frequency drift
when a readout tone is active. In transmon qubits the readout tone (resonator-mediated)
shifts the qubit frequency by a calculable amount and must be corrected in gate
calibrations (the dispersive shift $\chi$ is the dominant term).*


In [ ]:
# Compute dressed 0->1 energy gap vs drive amplitude
epsilon_arr_GHz = np.linspace(0, 0.5 * K_GHz, 40)   # sweep ε from 0 to 0.5K
omega01_dressed = np.zeros(len(epsilon_arr_GHz))

for i, eps_ghz in enumerate(epsilon_arr_GHz):
    Delta_fixed = TWO_PI * (-0.005)    # keep detuning fixed at -5 MHz
    H_rwa_i = (-Delta_fixed * n_op
               - (TWO_PI * K_GHz / 2) * adag * adag * a * a
               + (TWO_PI * eps_ghz / 2) * (a + adag))
    evals_i = H_rwa_i.eigenenergies()
    # The dressed gap (in rotating frame) between the two lowest-energy eigenstates
    omega01_dressed[i] = (evals_i[1] - evals_i[0]) / TWO_PI  # GHz

omega01_0 = omega01_dressed[0]  # undriven gap

# Second-order perturbation theory prediction (harmonic limit)
Delta_GHz = -0.005   # GHz
omega01_2nd_order = omega01_0 + (epsilon_arr_GHz**2 / 4) / Delta_GHz

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(epsilon_arr_GHz * 1e3, (omega01_dressed - omega01_0) * 1e3,
        "C0-o", ms=4, lw=1.5, label="Exact dressed gap (RWA Hamiltonian)")
ax.plot(epsilon_arr_GHz * 1e3, (omega01_2nd_order - omega01_0) * 1e3,
        "C1--", lw=1.5, label=r"2nd-order: $\varepsilon^2/4\Delta$")
ax.axhline(0, color="gray", lw=0.8, ls=":")
ax.set_xlabel(r"Drive amplitude $\varepsilon/2\pi$ (MHz)")
ax.set_ylabel(r"AC Stark shift $\delta\omega_{01}/2\pi$ (MHz)")
ax.set_title(
    rf"AC Stark shift vs. drive amplitude "
    rf"($\Delta/2\pi = {Delta_GHz*1e3:.0f}$ MHz, $K/2\pi = {K_GHz*1e3:.0f}$ MHz)"
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Where does Kerr start to matter?
eps_K = 0.5 * K_GHz
idx_K = np.argmin(np.abs(epsilon_arr_GHz - eps_K))
print(f"At ε = 0.5K = {eps_K*1e3:.0f} MHz:")
print(f"  Exact AC Stark shift = {(omega01_dressed[idx_K]-omega01_0)*1e3:.3f} MHz")
print(f"  2nd-order prediction  = {(omega01_2nd_order[idx_K]-omega01_0)*1e3:.3f} MHz")
print(f"  Kerr correction       = {((omega01_dressed[idx_K]-omega01_2nd_order[idx_K]))*1e3:.3f} MHz")


The blue curve shows the exact dressed gap computed from the RWA Hamiltonian;
the dashed orange line is the second-order perturbation theory result
$\delta\omega \approx \epsilon^2 / (4\Delta)$.
At weak drive they agree; at $\epsilon \sim K$ the Kerr term generates higher-order
corrections that curve the exact result away from the parabola.

**Physical mechanism:**
The drive admixes a small component of $|1\rangle$ into $|0\rangle$ and vice versa.
This dressing lowers the energy of the ground state by $\epsilon^2 / (4|\Delta|)$
(for $\Delta < 0$) and raises the excited state similarly, increasing the gap.
With Kerr, the mixing is no longer symmetric between levels, leading to corrections
that are higher order in $\epsilon / (K, \Delta)$.


<a id="sec5"></a>
## 5  Exercises

### Exercise 1: RWA validity at two different drive frequencies

Reproduce the Exact vs. RWA comparison from Section 3 for two cases:
1. **Resonant drive:** $\omega_d = \omega_0$ (zero detuning)
2. **Far-detuned drive:** $\omega_d = \omega_0 - K$ (driving the $|1\rangle\to|2\rangle$ transition instead)

For each case plot $\langle\hat{n}\rangle(t)$ for both exact and RWA, and compute the
maximum residual $|\text{Exact} - \text{RWA}|$.

**Expected result:**
- Resonant drive: large population build-up, larger absolute residual but same small *relative* error.
- Far-detuned drive: small average population, but the RWA residual should be similar in magnitude
  to the resonant case (set by $\epsilon / 2\omega_d$, not by the detuning).


In [ ]:
# YOUR CODE HERE
# Hint: use cfg.replace(omega_d=...) to set a new drive frequency
# then rebuild H_exact = make_H_drive_td(cfg_new)
# and H_rwa with the updated Delta = cfg_new.omega_d - cfg_new.omega0


### Exercise 2: Frequency of the counter-rotating oscillation

The counter-rotating residual $\delta\langle\hat{n}\rangle(t)$ oscillates at a
well-defined frequency.  From the lab-frame Hamiltonian $\hat{H}(t)$, identify
which term produces this oscillation and predict its frequency analytically.

Then take the Fourier transform of the residual computed in Section 3 and verify that
the dominant frequency matches your prediction.

**Hint:** write $\cos(\omega_d t) = (e^{i\omega_d t} + e^{-i\omega_d t})/2$
and collect terms that end up oscillating as $e^{\pm 2i\omega_d t}$ after the rotating-frame
transformation.


In [ ]:
# Solution (hidden in PDF output)
# The counter-rotating terms oscillate at 2*omega_d in the lab frame.
# After the rotating frame transform, they appear at 2*omega_d in the equations of motion.
# Fourier transform of residual should peak at 2*omega_d / (2*pi) = 2 * omega_d_GHz.

dt = tlist[1] - tlist[0]
freqs = np.fft.rfftfreq(len(tlist), d=dt)  # Hz (natural units of 1/s)
fft_res = np.abs(np.fft.rfft(residual))

fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(freqs / 1e9, fft_res, lw=1.2, color="C2")
ax.axvline(2 * omega0_GHz, color="C1", ls="--", lw=1.5,
           label=rf"$2\omega_d/2\pi = {2*omega0_GHz:.1f}$ GHz")
ax.set_xlim(0, 15)
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("|FFT|")
ax.set_title("Fourier transform of counter-rotating residual")
ax.legend()
plt.tight_layout()
plt.show()

# Find peak
peak_freq = freqs[np.argmax(fft_res)] / 1e9
print(f"Peak frequency in residual FFT: {peak_freq:.2f} GHz")
print(f"Expected (2*omega_d/2pi): {2*omega0_GHz:.2f} GHz")
